In [ ]:
# this notebook conduct the following task:
# 1 Extract Hchain Lchain and antigen chain from the csv info of # https://opig.stats.ox.ac.uk/webapps/sabdab-sabpred/sabdab/search/?all=true
#   structures_dir = "/content/drive/MyDrive/@antibody/antigen_epitope/data/structures/"
# 2 Extract antigen's structure info and interaction info, based on the antibody-antigen complex (two chains for antibody, one chain for antigen) from the previous step,
#   output_dir = "/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_info/"
# 3 TEST parts, use pymol to calculate the secondary structure and SaSaRate

In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive',force_remount=True)

In [ ]:
## for the first-time execution of this code
## install the packages in google Drive locally (/content/drive/MyDrive/Colab Notebooks/antigen_antibody/packages/)
import os
import sys

nb_path = '/content/notebooks'

if not os.path.exists(nb_path):
    os.symlink('/content/drive/MyDrive/@antibody/antigen_epitope/packages', nb_path)

sys.path.insert(0, nb_path)

In [ ]:
# first-time use execution
!pip install biopython --target=$nb_path

In [ ]:
!pip install scipy --target=$nb_path

In [ ]:
!pip install pywavefront --target=$nb_path

In [ ]:
!pip install ipyvolume --target=$nb_path

In [ ]:
!pip install pyvista --target=$nb_path

In [ ]:
!pip install matplotlib ipywidgets --target=$nb_path

In [ ]:
!apt-get install -y python3-pymol

In [ ]:
# INSTALL pymesh!!
# upload the .whl file for installation
# https://github.com/nuvolos-cloud/PyMesh/releases
from google.colab import files
uploaded = files.upload()
## install pymesh to calculate the curvature of the protein surface
!pip install pymesh2-0.3.1-cp310-cp310-linux_x86_64.whl --upgrade

In [ ]:
## non-first-time execution
## package was previously installed through process_data_positive_patch_3points-Graph.ipynb
import sys
sys.path.append('/content/drive/MyDrive/@antibody/antigen_epitope/packages')

In [ ]:
import os
import numpy as np
import pandas as pd
from Bio import PDB

In [ ]:
#@title Read csv file
# https://opig.stats.ox.ac.uk/webapps/sabdab-sabpred/sabdab/search/?all=true
# oneNote location: antigen_epitope

# Define the data directory
data_dir = '/content/drive/MyDrive/@antibody/antigen_epitope/data/'

# File name
file_name = 'sabdab_protein_summary.tsv'
file_path = data_dir + file_name

# Read the CSV file into a pandas DataFrame
df = pd.read_csv(file_path, delimiter='\t')

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

print(df.shape)
# Show the first few rows of the DataFrame
df.head(5)

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [ ]:
# Remove rows where 'antigen_chain' is NaN
# Remove rows where 'antigen_chain' is NaN or empty
df = df.dropna(subset=['antigen_chain'])
df = df[df['antigen_chain'].str.strip() != '']
filtered_df = df.dropna(subset=['antigen_chain'])

print(filtered_df.shape)

In [ ]:
df= df.head(5)
df

In [ ]:
#@title  Extract Hchain Lchain and antigen chain from the csv info
# if Hchain and Lchain have the same id, only extrace once
# each complex has 3 or 2 chains, one is antibody, the other is antigen
# complexes are saved here:/content/drive/MyDrive/@antibody/antigen_epitope/data/structures/

from Bio import PDB
import pandas as pd
import os

# Initialize PDB parser and retriever
parser = PDB.PDBParser(QUIET=True)
pdbl = PDB.PDBList()

# Create a DataFrame from your data; replace this with your actual DataFrame
# df = ...

# Filter the DataFrame to include only necessary columns and drop rows with NA in 'antigen_chain'
filtered_df = filtered_df[['pdb', 'Hchain', 'Lchain', 'model', 'antigen_chain', 'antigen_type']].dropna(subset=['antigen_chain'])

# Create a temporary directory for downloaded PDB files
os.makedirs("/content/tmp", exist_ok=True)

# Iterate through each row in the DataFrame
for index, row in filtered_df.iterrows():
    try:
        pdb_id = row['pdb']
        H_chain = row['Hchain'].strip()
        L_chain = row.get('Lchain')
        if L_chain:
            L_chain = L_chain.strip()

        antigen_chains = [chain.strip() for chain in row['antigen_chain'].split('|')]
        antigen_types = [type_.strip() for type_ in row['antigen_type'].split('|')]

        selected_antigen_chains = [chain for chain, type_ in zip(antigen_chains, antigen_types) if type_ == 'protein']

        downloaded_file_path = pdbl.retrieve_pdb_file(pdb_id, pdir='/content/tmp', file_format='pdb', overwrite=True)
        if not downloaded_file_path:
            print(f"Desired structure {pdb_id} doesn't exist")
            continue

        print(f"Downloading PDB structure '{pdb_id}'...")

        structure = parser.get_structure(pdb_id, downloaded_file_path)

        for antigen_chain in selected_antigen_chains:
            print(f"Processing antigen chain {antigen_chain}")

            io = PDB.PDBIO()
            io.set_structure(structure)

            antibody_chains = [H_chain]
            if L_chain and L_chain != H_chain:
                antibody_chains.append(L_chain)

            class ChainSelector(PDB.Select):
                def accept_chain(self, chain):
                    is_antigen = chain.id.strip() == antigen_chain
                    is_antibody = chain.id.strip() in antibody_chains
                    is_accepted = is_antigen or is_antibody
                    print(f"Chain {chain.id} {'accepted' if is_accepted else 'rejected'}")
                    return is_accepted

            L_str = f"_{L_chain}" if L_chain else "_nan"
            output_path = f"/content/drive/MyDrive/@antibody/antigen_epitope/data/structures/{pdb_id}_{H_chain}{L_str}_{antigen_chain}.pdb"

            io.save(output_path, ChainSelector())

    except Exception as e:
        print(f"An error occurred while processing {pdb_id}: {e}")


In [ ]:
# in the directory /content/drive/MyDrive/@antibody/antigen_epitope/data/structures/

In [ ]:
# Not need to change some file names of the complex
# e.g.   7uoj_k_m_I(1).pdb into  7uoj_k_m_I.pdb
# in colab it shows: 7uoj_K_M_I.pdb 7uoj_k_m_I.pdb

In [ ]:
#@title Save antigen as .pdb (run this code if antigen pdbs are not created)
antigen_pdb_dir = '/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_pdbs/'
if not os.path.exists(antigen_pdb_dir):
  os.makedirs(antigen_pdb_dir)
  print(f"Folder '{antigen_pdb_dir}' was created.")
else:
  print(f"Folder '{antigen_pdb_dir}' already exists.")

structures_dir = "/content/drive/MyDrive/@antibody/antigen_epitope/data/structures/"
output_dir = "/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_info_SI/"

for pdb_file in os.listdir(structures_dir):
  if pdb_file.endswith('.pdb'):
    pdb_file_path = os.path.join(structures_dir, pdb_file)

    # Extract antigen chain information
    antigen_chain = pdb_file.split('_')[-1][0]

    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_file.split('.')[0], pdb_file_path)

    io = PDB.PDBIO()
    # save antigen pdb file
    for model in structure:
      for chain in model:
        if chain.id == antigen_chain:
          io.set_structure(chain)
          output_filename = f"{pdb_file.split('.')[0]}_antigen.pdb"
          io.save(antigen_pdb_dir + output_filename)

In [ ]:
#@title Create .obj file for each antigen pdb (run this code if antigen objs are not created)
# The obj file could be large (40mb per file)
from pymol import cmd

antigen_obj_dir = '/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_objs/'
if not os.path.exists(antigen_obj_dir):
  os.makedirs(antigen_obj_dir)
  print(f"Folder '{antigen_obj_dir}' was created.")
else:
  print(f"Folder '{antigen_obj_dir}' already exists.")

for pdb_filename in os.listdir(antigen_pdb_dir):
  if pdb_filename.endswith('.pdb'):
    # reset pymol
    cmd.reinitialize()

    pdb_file_path = os.path.join(antigen_pdb_dir, pdb_filename)

    # load pdb
    cmd.load(pdb_file_path, 'pdb_filename')
    cmd.hide('everything', 'pdb_filename')
    cmd.show('surface', 'pdb_filename')

    # set camera view
    cmd.set_view((
        1,    0,    0,
        0,    1,    0,
        0,    0,    1,
        0,    0,    0,
        0,    0,    0,
        0,    300,  1
    ))

    # save obj file
    obj_filename = pdb_filename.replace('.pdb', '.obj')
    cmd.save(os.path.join(antigen_obj_dir, obj_filename), 'pdb_filename', format='obj')

In [ ]:
import ipyvolume as ipv
from matplotlib import cm
import pyvista as pv
import pymesh

In [ ]:
# dir to save info with SI
SI_output_dir = "/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_info_SI/"

antigen_obj_dir = '/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_objs/'
antigen_pdb_dir = '/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_pdbs/'

In [ ]:
#@title Process .obj files
import pywavefront

for obj_filename in os.listdir(antigen_obj_dir):
  if obj_filename.endswith('.obj'):

    # Read the .obj file
    mesh = pywavefront.Wavefront(antigen_obj_dir + obj_filename, collect_faces=True)
    obj_vertices = set(mesh.vertices)
    # print('file name: ', obj_filename)
    # print('obj verices number: ', len(obj_vertices))
    obj_coords = np.array([v for v in set(obj_vertices)])
    obj_mean_curvature = np.array([np.nan] * len(obj_coords))
    obj_guassian_curvature = np.array([np.nan] * len(obj_coords))
    obj_curvature = np.array([np.nan] * len(obj_coords))


    # 15 seems to be the best neighbor number
    n_neighbors = 15

    # Calculate centroid by mean
    centroid = np.mean(obj_coords, axis=0)

    for idx in range(len(obj_coords)):
      current_coords = obj_coords[idx]
      dist_coords = obj_coords - current_coords

      dists = np.linalg.norm(dist_coords, axis=1)
      sorted_indices = np.argsort(dists)

      nearest_neighbors_idx = np.argsort(dists)[1:(n_neighbors + 1)]
      vertices = dist_coords[np.concatenate([[idx], nearest_neighbors_idx]), :]

      cloud = pv.PolyData(vertices)
      triangulated = cloud.delaunay_2d()
      shell = triangulated.extract_geometry()

      extracted_vertices = np.array(shell.points)
      extracted_faces = np.array(shell.faces).reshape(-1, 4)[:, 1:]
      extracted_faces = np.array([face for face in extracted_faces if 0 in face])

      mesh = pymesh.form_mesh(extracted_vertices, extracted_faces)
      mesh, info = pymesh.remove_isolated_vertices(mesh)

      mesh.add_attribute("vertex_mean_curvature")
      H = mesh.get_attribute("vertex_mean_curvature")
      mean_H = H[0]

      mesh.add_attribute("vertex_gaussian_curvature")
      K = mesh.get_attribute("vertex_gaussian_curvature")
      gaussian_K = K[0]

      elem = np.square(mean_H) - gaussian_K

      if elem < 0:
        elem = 1e-8
      elem = np.array(elem)

      k1 = mean_H + np.sqrt(elem)
      k2 = mean_H - np.sqrt(elem)

      si = (k1 + k2) / (k1 - k2)
      si = np.arctan(si) * (-2 / np.pi)

      # adjust curvature sign based on average direction to centroid
      dist_to_centroid = np.linalg.norm(current_coords - centroid)
      neighbor_dists_to_centroid = [np.linalg.norm(obj_coords[i] - centroid) for i in nearest_neighbors_idx]
      avg_neighbor_dist_to_centroid = np.mean(neighbor_dists_to_centroid)

      curvature_sign_by_distance = -1 if dist_to_centroid < avg_neighbor_dist_to_centroid else 1

      curvature_sign = curvature_sign_by_distance

      if curvature_sign > 0:
        si = abs(si)
      elif curvature_sign < 0:
        si = -abs(si)

      obj_mean_curvature[idx] = mean_H
      obj_guassian_curvature[idx] = gaussian_K
      obj_curvature[idx] = si


    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure('protein', antigen_pdb_dir + obj_filename.replace('obj', 'pdb'))

    # select antigen chain name
    antigen_chain = obj_filename.split('_')[-2][0]
    model = structure[0]
    chain = model[antigen_chain]

    # Remove HETATM atom and select C-alpha
    atoms = [atom for atom in chain.get_atoms() if atom.get_parent().id[0] == " " and atom.name=='CA']
    atom_curvature = [None] * len(atoms)

    atom_coords = np.array([atom.coord for atom in atoms])
    # print('ATOM (CA) number: ', len(atoms))


    # Calculate the atom curvature based on the average curvature of nearest 10 points
    n_neighbors = 10
    for idx in range(len(atom_coords)):
      current_atom_coords = atom_coords[idx]
      dist_coords = obj_coords - current_atom_coords
      dists = np.linalg.norm(dist_coords, axis=1)

      nearest_neighbors_idx = np.argsort(dists)[:n_neighbors]
      atom_curvature[idx] = np.mean(obj_curvature[nearest_neighbors_idx])

    # dataframe
    columns = ['name', 'ResID', 'AA', 'SI']
    df = pd.DataFrame(columns=columns)
    df['ResID'] = [atom.get_full_id()[3][1] for atom in atoms]
    df['name'] = obj_filename.split('.')[0].replace('_antigen','')
    df['AA'] = [atom.get_parent().get_resname() for atom in atoms]
    df['SI'] = atom_curvature

    output_path = os.path.join(SI_output_dir, f"{obj_filename.split('.')[0]}_info_SI.tsv")
    df.to_csv(output_path, sep='\t', index=False)

    print(f"Processed {obj_filename} and saved to {output_path}")

In [ ]:
#@title Extract antigen's structure info and interaction info ==> minumn atom-atom distance between sidechains

from Bio import PDB
import pandas as pd
from pymol import cmd, stored
from scipy.spatial import distance
import os

# Function to calculate center of mass of sidechain
# Glycine (GLY) does not have a side chain beyond its alpha carbon, so its side chain center of mass would be undefined in your code
def calculate_center_of_mass(residue):
    if residue.resname == "GLY":
        return residue['CA'].coord  # Return alpha carbon coordinates for Glycine

    mass = 0
    x, y, z = 0, 0, 0
    backbone_atoms = ['N', 'CA', 'C', 'O']
    for atom in residue.get_unpacked_list():
        if atom.name not in backbone_atoms:
            mass += atom.mass
            x += atom.coord[0] * atom.mass
            y += atom.coord[1] * atom.mass
            z += atom.coord[2] * atom.mass
    if mass == 0:
        return 0, 0, 0
    return x / mass, y / mass, z / mass

# Function to check for heavy atom-atom interactions
def check_atom_atom_interaction(residue1, residue2, cutoff_distance=4.5, ca_cutoff_distance=15):
    # To save computational time, Check the CA-CA distance first
    if "CA" in residue1 and "CA" in residue2:
        ca_distance = distance.euclidean(residue1["CA"].coord, residue2["CA"].coord)
        if ca_distance > ca_cutoff_distance:
            return False  # If the CA-CA distance is greater than the threshold, no need to check atom-atom distances

    # Proceed with atom-atom distance calculations
    for atom1 in residue1.get_unpacked_list():
        if atom1.element == "H":  # skip hydrogen atoms
            continue
        for atom2 in residue2.get_unpacked_list():
            if atom2.element == "H":  # skip hydrogen atoms
                continue
            if distance.euclidean(atom1.coord, atom2.coord) <= cutoff_distance:
                return True  # found an interaction
    return False



# Process each PDB file in the directory
structures_dir = "/content/drive/MyDrive/@antibody/antigen_epitope/data/structures/"
output_dir = "/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_info/"

for pdb_file in os.listdir(structures_dir):
    if pdb_file.endswith('.pdb'):
        pdb_file_path = os.path.join(structures_dir, pdb_file)

        # Extract chain information
        chains = pdb_file.split('_')[1:-1]  # Extract the chain names (e.g., ['H', 'L'] or ['S'])

        antigen_chain = pdb_file.split('_')[-1][0]  # Extract antigen chain (e.g., 'X')

        # Load the PDB file into PyMOL
        cmd.reinitialize()  # Clear previous data
        cmd.load(pdb_file_path, "protein_complex")

        # Calculate relative SASA in the presence of antibody
        # SaSaRate = cmd.get_sasa_relative("protein_complex and chain " + antigen_chain)
        # sasa_values = [value for key, value in SaSaRate.items()]

        # Initialize an empty list for secondary structure
        stored.ss_values = []
        cmd.iterate("protein_complex and chain " + antigen_chain + " and name CA", "stored.ss_values.append(ss)")

         # Hide or remove the antibody chains
        cmd.remove("chain " + "+".join(chains))  # Using the `remove` command to delete the antibody chains
        # Calculate relative SASA for the antigen chain in the absence of the antibody
        SaSaRate = cmd.get_sasa_relative("protein and chain " + antigen_chain)
        sasa_values = [value for key, value in SaSaRate.items()]

        # Initialize PDB Parser
        parser = PDB.PDBParser(QUIET=True)

        # Load structure
        structure = parser.get_structure(pdb_file.split('.')[0], pdb_file_path)

        # Initialize DataFrame
        columns = ['name', 'ResID', 'AA', 'SS', 'Cax', 'Cay', 'Caz', 'SCx', 'SCy', 'SCz', 'SaSaRate',
                   'G', 'A', 'V', 'I', 'L', 'S', 'T', 'D', 'N', 'E', 'Q', 'K', 'R', 'C', 'M', 'F', 'Y', 'W', 'H', 'P']
        df = pd.DataFrame(columns=columns)

        # Prepare a list for antibody residues
        antibody_residues = []
        for model in structure:
            for chain in model:
                if chain.id in chains:
                    for residue in chain:
                        # Ensure that the residue is an amino acid
                        if residue.resname in PDB.Polypeptide.protein_letters_3to1:
                            antibody_residues.append(residue)

        # Iterate through structure to get antigen residues
        for model in structure:
            for chain in model:
                if chain.id == antigen_chain:
                    for i, residue in enumerate(chain):
                        if "CA" in residue:
                            c_alpha = residue['CA'].coord
                            # Safeguard against IndexError
                            ss = stored.ss_values[i] if i < len(stored.ss_values) else 'Unknown'
                            sc_x, sc_y, sc_z = calculate_center_of_mass(residue)

                            interaction_data = {aa: 0 for aa in ['G', 'A', 'V', 'I', 'L', 'S', 'T', 'D', 'N', 'E', 'Q', 'K', 'R', 'C', 'M', 'F', 'Y', 'W', 'H', 'P']}
                            for ab_residue in antibody_residues:
                                if check_atom_atom_interaction(residue, ab_residue):
                                    one_letter_code = PDB.Polypeptide.protein_letters_3to1[ab_residue.resname]
                                    interaction_data[one_letter_code] = 1

                            data = {'name': pdb_file.split('.')[0],
                                    'ResID': residue.id[1],  # Residue number
                                    'AA': residue.resname,
                                    'SS': ss,
                                    'Cax': c_alpha[0],
                                    'Cay': c_alpha[1],
                                    'Caz': c_alpha[2],
                                    'SCx': sc_x,
                                    'SCy': sc_y,
                                    'SCz': sc_z,
                                    'SaSaRate': sasa_values[i]}
                            data.update(interaction_data)
                            df = pd.concat([df, pd.DataFrame([data])], ignore_index=True)

        # Save DataFrame to TSV
        output_path = os.path.join(output_dir, f"{pdb_file.split('.')[0]}_info.tsv")
        df.to_csv(output_path, sep='\t', index=False)

        print(f"Processed {pdb_file} and saved to {output_path}")

In [ ]:
#@title Extract antigen's structure info and interaction info with COM-COM distance between sidechains

from Bio import PDB
import pandas as pd
from pymol import cmd, stored
from scipy.spatial import distance
import os

# Function to calculate center of mass of sidechain
# Glycine (GLY) does not have a side chain beyond its alpha carbon, so its side chain center of mass would be undefined in your code
def calculate_center_of_mass(residue):
    if residue.resname == "GLY":
        return residue['CA'].coord  # Return alpha carbon coordinates for Glycine

    mass = 0
    x, y, z = 0, 0, 0
    backbone_atoms = ['N', 'CA', 'C', 'O']
    for atom in residue.get_unpacked_list():
        if atom.name not in backbone_atoms:
            mass += atom.mass
            x += atom.coord[0] * atom.mass
            y += atom.coord[1] * atom.mass
            z += atom.coord[2] * atom.mass
    if mass == 0:
        return 0, 0, 0
    return x / mass, y / mass, z / mass


# Process each PDB file in the directory
structures_dir = "/content/drive/MyDrive/@antibody/antigen_epitope/data/structures/"
output_dir = "/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_info/"

for pdb_file in os.listdir(structures_dir):
    if pdb_file.endswith('.pdb'):
        pdb_file_path = os.path.join(structures_dir, pdb_file)

        # Extract chain information
        chains = pdb_file.split('_')[1:-1]  # Extract the chain names (e.g., ['H', 'L'] or ['S'])
        antigen_chain = pdb_file.split('_')[-1][0]  # Extract antigen chain (e.g., 'X')

        # Load the PDB file into PyMOL
        cmd.reinitialize()  # Clear previous data
        cmd.load(pdb_file_path, "protein_complex")

        # Calculate relative SASA
        SaSaRate = cmd.get_sasa_relative("protein_complex and chain " + antigen_chain)
        sasa_values = [value for key, value in SaSaRate.items()]

        # Initialize an empty list for secondary structure
        stored.ss_values = []
        cmd.iterate("protein_complex and chain " + antigen_chain + " and name CA", "stored.ss_values.append(ss)")

        # Initialize PDB Parser
        parser = PDB.PDBParser(QUIET=True)

        # Load structure
        structure = parser.get_structure(pdb_file.split('.')[0], pdb_file_path)

        # Initialize DataFrame
        columns = ['name', 'ResID', 'AA', 'SS', 'Cax', 'Cay', 'Caz', 'SCx', 'SCy', 'SCz', 'SaSaRate',
                   'G', 'A', 'V', 'I', 'L', 'S', 'T', 'D', 'N', 'E', 'Q', 'K', 'R', 'C', 'M', 'F', 'Y', 'W', 'H', 'P']
        df = pd.DataFrame(columns=columns)

        # Prepare a list for antibody sidechain centers
        antibody_centers = []

        # Populate antibody_centers
        for model in structure:
            for chain in model:
                if chain.id in chains:
                    for residue in chain:
                        if "CA" in residue:
                            sc_x, sc_y, sc_z = calculate_center_of_mass(residue)
                            ab_resname = residue.resname
                            antibody_centers.append((ab_resname, (sc_x, sc_y, sc_z)))

        # Iterate through structure to get antigen residues
        for model in structure:
            for chain in model:
                if chain.id == antigen_chain:
                    for i, residue in enumerate(chain):
                        if "CA" in residue:
                            c_alpha = residue['CA'].coord
                            sc_x, sc_y, sc_z = calculate_center_of_mass(residue)
                            #ss = stored.ss_values[i]
                            # Safeguard against IndexError
                            ss = stored.ss_values[i] if i < len(stored.ss_values) else 'Unknown'

                            interaction_data = {aa: 0 for aa in ['G', 'A', 'V', 'I', 'L', 'S', 'T', 'D', 'N', 'E', 'Q', 'K', 'R', 'C', 'M', 'F', 'Y', 'W', 'H', 'P']}
                            antigen_center = (sc_x, sc_y, sc_z)

                            for ab_resname, ab_center in antibody_centers:
                                if distance.euclidean(antigen_center, ab_center) <= 7:
                                  try:
                                    one_letter_code = PDB.Polypeptide.protein_letters_3to1[ab_resname]
                                    interaction_data[one_letter_code] = 1
                                  except KeyError:
                                    print(f"Warning: Unexpected residue name '{ab_resname}' encountered.")

                            data = {'name': pdb_file.split('.')[0],
                                    'ResID': residue.id[1],  # Residue number
                                    'AA': residue.resname,
                                    'SS': ss,
                                    'Cax': c_alpha[0],
                                    'Cay': c_alpha[1],
                                    'Caz': c_alpha[2],
                                    'SCx': sc_x,
                                    'SCy': sc_y,
                                    'SCz': sc_z,
                                    'SaSaRate': sasa_values[i]}
                            data.update(interaction_data)
                            df = pd.concat([df, pd.DataFrame([data])], ignore_index=True)

        # Save DataFrame to TSV
        output_path = os.path.join(output_dir, f"{pdb_file.split('.')[0]}_info.tsv")
        df.to_csv(output_path, sep='\t', index=False)

        print(f"Processed {pdb_file} and saved to {output_path}")



In [ ]:
# how many tsv files processed
import os

output_dir = "/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_info/"

# List the files in the directory
files_in_directory = os.listdir(output_dir)

# Count the number of .tsv files
number_of_tsv_files = len([f for f in files_in_directory if f.endswith('.tsv')])

print(f"There are {number_of_tsv_files} .tsv files in {output_dir}.")


In [ ]:
import pandas as pd
import numpy as np
import pymesh

# Read the TSV file into a DataFrame
file_path = "/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_curvature_testing/1a2y_B_A_C_info.tsv"
result_df = pd.read_csv(file_path, sep='\t')

# List of thresholds
thresholds = [0.1, 0.2, 0.3]

for threshold in thresholds:

    # Filter the DataFrame to only include residues with SaSaRate > threshold
    surface_residues = result_df[result_df['SaSaRate'] > threshold]

    # Initialize an empty list to store curvature
    curvatures = [None] * len(result_df)

    # Loop over each surface residue to find its 3 nearest neighbors
    for idx, residue in surface_residues.iterrows():
        coords = result_df[['SCx', 'SCy', 'SCz']].values.astype(float)
        residue_coords = np.array(residue[['SCx', 'SCy', 'SCz']].values, dtype=float)
        dists = np.linalg.norm(coords - residue_coords, axis=1)
        nearest_neighbors_idx = np.argsort(dists)[1:4]  # Exclude itself

        # Create vertices (coordinates of the residue and its neighbors)
        vertices = coords[np.concatenate([[idx], nearest_neighbors_idx]), :]

        # Create faces (assuming the points form a tetrahedron)
        faces = np.array([[0, 1, 2], [0, 2, 3], [0, 3, 1], [1, 2, 3]])

        # Create mesh in PyMesh
        mesh = pymesh.form_mesh(vertices, faces)

        # Calculate Gaussian curvature
        mesh.add_attribute("vertex_gaussian_curvature")
        curvature = mesh.get_attribute("vertex_gaussian_curvature")

        # Store the average curvature
        curvatures[idx] = np.mean(curvature)

    # Add the curvature column to the DataFrame
    result_df['Curvature'] = curvatures

    # Save the updated DataFrame back to a new TSV file
    result_df.to_csv(f'/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_curvature_testing/1a2y_B_A_C_info_with_curvature_{threshold}.tsv', sep='\t', index=False)


In [ ]:
#@title TEST -- Methods_1 part_1 How to calculate the curvature info for data frame

# INSTALL pymesh!!
# upload the .whl file for installation
# https://github.com/nuvolos-cloud/PyMesh/releases
from google.colab import files
uploaded = files.upload()

## install pymesh to calculate the curvature of the protein surface
!pip install pymesh2-0.3.1-cp310-cp310-linux_x86_64.whl --target=$nb_path  --upgrade

In [ ]:
import pymesh

In [ ]:
import pandas as pd
import numpy as np
import pymesh

def calculate_curvature(df, n_neighbors=3, threshold=0.1):
    # Filter based on threshold
    surface_residues = df[df['SaSaRate'] > threshold]

    # Initialize empty list to store curvature
    curvatures = [None] * len(df)

    for idx, residue in surface_residues.iterrows():
        coords = df[['SCx', 'SCy', 'SCz']].values.astype(float)
        residue_coords = np.array(residue[['SCx', 'SCy', 'SCz']].values, dtype=float)

        # Calculate distances and get indices of nearest neighbors
        dists = np.linalg.norm(coords - residue_coords, axis=1)
        nearest_neighbors_idx = np.argsort(dists)[1:(n_neighbors + 1)]  # Exclude itself

        # Create vertices and faces
        vertices = coords[np.concatenate([[idx], nearest_neighbors_idx]), :]
        faces = [[0, i, i + 1] for i in range(1, n_neighbors)]  # Create faces assuming they form a convex hull with the residue

        # Convert list to numpy array
        faces = np.array(faces)

        # Create mesh
        mesh = pymesh.form_mesh(vertices, faces)

        # Calculate Gaussian curvature
        # mesh.add_attribute("vertex_gaussian_curvature")
        # curvature = mesh.get_attribute("vertex_gaussian_curvature")

        # Store the average curvature
        # curvatures[idx] = np.mean(curvature)

        mesh.add_attribute("vertex_mean_curvature")
        H = mesh.get_attribute("vertex_mean_curvature")
        mean_H = np.mean(H)

        mesh.add_attribute("vertex_gaussian_curvature")
        K = mesh.get_attribute("vertex_gaussian_curvature")
        mean_K = np.mean(H)

        elem = np.square(mean_H) - mean_K
        # In some cases this equation is less than zero, likely due to the method that computes the mean and gaussian curvature.
        # set to an epsilon.

        elem = np.array(elem)
        elem[elem<0] = 1e-8
        k1 = mean_H + np.sqrt(elem)
        k2 = mean_H - np.sqrt(elem)
        # Compute the shape index
        si = (k1+k2)/(k1-k2)
        si = np.arctan(si)*(2/np.pi)

        curvatures[idx] = si

    return curvatures


# Read the TSV file into a DataFrame
file_path = "/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_curvature_testing/1a2y_B_A_C_info.tsv"
result_df = pd.read_csv(file_path, sep='\t')

# List of thresholds and neighbors
thresholds = [0.1]
n_neighbors_list = [12, 16, 20]

for threshold in thresholds:
    for n_neighbors in n_neighbors_list:
        curvatures = calculate_curvature(result_df, n_neighbors=n_neighbors, threshold=threshold)
        result_df['Curvature'] = curvatures

        # Save the updated DataFrame back to a new TSV file
        output_file_path = f"/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_curvature_testing/output_curvature_{threshold}_{n_neighbors}.tsv"
        result_df.to_csv(output_file_path, sep='\t', index=False)



In [ ]:
#@title TEST -- Method_1 part_2 add the curvature info to PDB as b-factor
from Bio.PDB import PDBParser, PDBIO
import pandas as pd

# List of thresholds and n_neighbors (should match the ones used in the first part)
thresholds = [0.1]
n_neighbors_list = [12, 16, 20]

for threshold in thresholds:
    for n_neighbors in n_neighbors_list:
        # Load the corresponding TSV file based on the threshold and n_neighbors
        curvature_df = pd.read_csv(f'/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_curvature_testing/output_curvature_{threshold}_{n_neighbors}.tsv', sep='\t')

        # Filter the DataFrame to include only the rows corresponding to chain C
        curvature_df_chain_c = curvature_df[curvature_df['name'] == '1a2y_B_A_C']

        # Create a dictionary for quick lookup of curvature values
        curvature_dict = curvature_df_chain_c.set_index('ResID')['Curvature'].to_dict()

        # Initialize a PDB parser
        parser = PDBParser(QUIET=True)

        # Load the structure
        pdb_file_path = '/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_curvature_testing/1a2y_B_A_C.pdb'
        structure = parser.get_structure('1a2y_B_A_C', pdb_file_path)

        # Create a new PDBIO object for saving the modified structure
        io = PDBIO()

        # Update B-factor with curvature information and filter to only include chain C
        def select_and_update(structure):
            for model in structure:
                for chain in model:
                    if chain.id == 'C':
                        for residue in chain:
                            res_id = residue.id[1]  # Get the residue number
                            if res_id in curvature_dict:
                                curvature_value = curvature_dict[res_id]
                                if curvature_value is not None:
                                    for atom in residue:
                                        atom.set_bfactor(curvature_value)
                        return chain

        # Save the updated chain C structure
        output_path = f'/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_curvature_testing/1a2y_B_A_C_chain_C_with_curvature_{threshold}_{n_neighbors}.pdb'
        io.set_structure(select_and_update(structure))
        io.save(output_path)


In [ ]:
#@title TEST generate mesh for all the atoms of pdb
import pandas as pd
import numpy as np
import pymesh
from Bio.PDB import PDBParser, PDBIO

def calculate_curvature(pdb_file_path, n_neighbors=3):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('protein', pdb_file_path)
    model = structure[0]
    chain = model['C']

    atoms = [atom for atom in chain.get_atoms()]
    coords = np.array([atom.coord for atom in atoms])
    curvatures = [None] * len(atoms)

    for idx, atom in enumerate(atoms):
        atom_coords = coords[idx]

        # Calculate distances and get indices of nearest neighbors
        dists = np.linalg.norm(coords - atom_coords, axis=1)
        nearest_neighbors_idx = np.argsort(dists)[1:(n_neighbors + 1)]

        # Create vertices and faces
        vertices = coords[np.concatenate([[idx], nearest_neighbors_idx]), :]
        faces = [[0, i, i + 1] for i in range(1, n_neighbors)]

        faces = np.array(faces)

        # Create mesh
        mesh = pymesh.form_mesh(vertices, faces)

        mesh.add_attribute("vertex_mean_curvature")
        H = mesh.get_attribute("vertex_mean_curvature")
        mean_H = np.mean(H)

        mesh.add_attribute("vertex_gaussian_curvature")
        K = mesh.get_attribute("vertex_gaussian_curvature")
        mean_K = np.mean(H)

        elem = np.square(mean_H) - mean_K
        elem = np.array(elem)
        elem[elem < 0] = 1e-8

        k1 = mean_H + np.sqrt(elem)
        k2 = mean_H - np.sqrt(elem)
        si = (k1 + k2) / (k1 - k2)
        si = np.arctan(si) * (2 / np.pi)

        curvatures[idx] = si

    return curvatures

# Function to update B-factor with curvature values
def select_and_update(structure, curvatures):
    for model in structure:
        for chain in model:
            if chain.id == 'C':
                for i, residue in enumerate(chain):
                    for j, atom in enumerate(residue):
                        if i * len(residue) + j < len(curvatures):
                            curvature_value = curvatures[i * len(residue) + j]
                            if curvature_value is not None:
                                atom.set_bfactor(curvature_value)

# Initialize a PDB parser
parser = PDBParser(QUIET=True)

# Load the structure
pdb_file_path = '/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_curvature_testing/1a2y_B_A_C.pdb'
structure = parser.get_structure('1a2y_B_A_C', pdb_file_path)

# Calculate curvatures
n_neighbors_list = [7]
for n_neighbors in n_neighbors_list:
    curvatures = calculate_curvature(pdb_file_path, n_neighbors=n_neighbors)

    # Create a new PDBIO object for saving the modified structure
    io = PDBIO()

    # Save the updated chain C structure
    output_path = f'/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_curvature_testing/1a2y_B_A_C_chain_C_with_curvature_{n_neighbors}.pdb'
    io.set_structure(structure)
    select_and_update(structure, curvatures)
    io.save(output_path)


In [ ]:
#@title TEST Method_2 pymol generate mesh object and then
from pymol import cmd

In [ ]:
from pymol import cmd



pdb_file_path = '/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_curvature_testing/1a2y_B_A_C.pdb'
obj_file_path = '/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_curvature_testing/model_chain_c.obj'  # Replace with your desired output path

# Load the PDB file into an object named 'protein'
cmd.load(pdb_file_path, 'protein')

# Create a selection for only the CA atoms of chain C
cmd.select('chain_c', 'chain C and polymer')

# Show the surface for the selected CA atoms
cmd.show('surface', 'chain_c')

# Save the view as a .png file
cmd.save(obj_file_path, 'chain C', format='obj')




In [ ]:
import pymesh
import numpy as np

# Load mesh
mesh = pymesh.load_mesh(obj_file_path)

# Enable attributes for curvature calculations
mesh.add_attribute("vertex_mean_curvature")
mesh.add_attribute("vertex_gaussian_curvature")

# Retrieve the curvature attributes
H = mesh.get_attribute("vertex_mean_curvature")
K = mesh.get_attribute("vertex_gaussian_curvature")

# Calculate mean curvature (H) and Gaussian curvature (K)
mean_H = np.mean(H)
mean_K = np.mean(K)

# Calculate principal curvatures k1 and k2
elem = np.square(mean_H) - mean_K
elem = np.array(elem)
elem[elem < 0] = 1e-8  # Handle negative values
k1 = mean_H + np.sqrt(elem)
k2 = mean_H - np.sqrt(elem)

# Compute the shape index for each vertex
si = (2/np.pi) * np.arctan((k1 + k2) / (k1 - k2))

# Initialize curvatures list
curvatures = np.zeros(mesh.num_vertices)

# Since we used the mean curvature and mean Gaussian curvature, the shape index would be the same for all vertices.
# Therefore, we populate the 'curvatures' list with the same shape index value.
curvatures.fill(si)

print(f"Curvatures: {curvatures}")


In [ ]:
len(H)

In [ ]:
# visualize the obj file
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from ipywidgets import interactive

def read_obj(filename):
    vertices = []
    faces = []
    with open(filename, 'r') as f:
        for line in f:
            parts = line.strip().split(" ")
            if parts[0] == "v":
                vertices.append([float(v) for v in parts[1:]])
            elif parts[0] == "f":
                faces.append([int(f.split('/')[0]) for f in parts[1:]])
    return np.array(vertices), np.array(faces)

# Replace with your .obj file path
obj_file_path = '/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_curvature_testing/model_chain_c.obj'
vertices, faces = read_obj(obj_file_path)

def plot_figure(elev, azim):
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    ax.view_init(elev=elev, azim=azim)

    for face in faces:
        for i in range(len(face)):
            ax.plot([vertices[face[i] - 1][0], vertices[face[i - 1] - 1][0]],
                    [vertices[face[i] - 1][1], vertices[face[i - 1] - 1][1]],
                    [vertices[face[i] - 1][2], vertices[face[i - 1] - 1][2]], 'b-')
    plt.show()

interactive_plot = interactive(plot_figure, elev=widgets.IntSlider(min=0, max=180, step=10, value=30), azim=widgets.IntSlider(min=0, max=360, step=10, value=30))
output = interactive_plot.children[-1]
interactive_plot


In [ ]:
import numpy as np

# Assuming mean_curvature and gaussian_curvature are your computed arrays
mean_curvature_clean = mean_curvature[np.isfinite(mean_curvature)]
gaussian_curvature_clean = gaussian_curvature[np.isfinite(gaussian_curvature)]

# Now compute the surface index
surface_index = np.mean(mean_curvature_clean) / np.mean(np.abs(gaussian_curvature_clean))
print("Approximate Surface Index:", surface_index)


In [ ]:
import numpy as np

# Initialize a boolean array to keep track of which vertices are used
vertex_used = np.zeros(mesh.num_vertices, dtype=bool)

# Mark used vertices
for face in mesh.faces:
    vertex_used[face] = True

# Find and count isolated vertices
isolated_vertices = np.where(vertex_used == False)[0]
print(f"Number of isolated vertices: {len(isolated_vertices)}")


In [ ]:
#@title TEST - generate the antigen info data frame with minimun Atom-Atom distance < 4.5
# this antigen data frame contains the structure info and interaction info

from Bio import PDB
import pandas as pd
from pymol import cmd, stored
from scipy.spatial import distance
import os

# Function to calculate center of mass of sidechain
def calculate_center_of_mass(residue):
    mass = 0
    x, y, z = 0, 0, 0
    backbone_atoms = ['N', 'CA', 'C', 'O']
    for atom in residue.get_unpacked_list():
        if atom.name not in backbone_atoms:
            mass += atom.mass
            x += atom.coord[0] * atom.mass
            y += atom.coord[1] * atom.mass
            z += atom.coord[2] * atom.mass
    if mass == 0:
        return 0, 0, 0
    return x / mass, y / mass, z / mass

# Function to check for heavy atom-atom interactions
def check_atom_atom_interaction(residue1, residue2, cutoff_distance=4.5):
    for atom1 in residue1.get_unpacked_list():
        if atom1.element == "H":  # skip hydrogen atoms
            continue
        for atom2 in residue2.get_unpacked_list():
            if atom2.element == "H":  # skip hydrogen atoms
                continue
            if distance.euclidean(atom1.coord, atom2.coord) <= cutoff_distance:
                return True  # found an interaction
    return False

# Load the PDB file into PyMOL
pdb_file_path = "/content/drive/MyDrive/@antibody/antigen_epitope/data/structures/1ors_B_A_C.pdb"
cmd.load(pdb_file_path, "protein_complex")

# Calculate relative SASA
SaSaRate = cmd.get_sasa_relative("protein_complex and chain C")
sasa_values = [value for key, value in SaSaRate.items()]

# Initialize an empty list for secondary structure
stored.ss_values = []
cmd.iterate("protein_complex and chain C and name CA", "stored.ss_values.append(ss)")

# Initialize PDB Parser
parser = PDB.PDBParser(QUIET=True)

# Load structure
structure = parser.get_structure('1ors_B_A_C', pdb_file_path)

# Initialize DataFrame
columns = ['name', 'ResID', 'AA', 'SS', 'Cax', 'Cay', 'Caz', 'SCx', 'SCy', 'SCz', 'SaSaRate',
           'G', 'A', 'V', 'I', 'L', 'S', 'T', 'D', 'N', 'E', 'Q', 'K', 'R', 'C', 'M', 'F', 'Y', 'W', 'H', 'P']
df = pd.DataFrame(columns=columns)

# Prepare a list for antibody sidechain centers
antibody_residues = []

# Populate antibody_residues
for model in structure:
    for chain in model:
        if chain.id in ["B", "A"]:
            for residue in chain:
                # Ensure that the residue is an amino acid
                if residue.resname in PDB.Polypeptide.protein_letters_3to1:
                    antibody_residues.append(residue)

# Iterate through structure to get antigen residues
for model in structure:
    for chain in model:
        if chain.id == "C":
            for i, residue in enumerate(chain):
                if "CA" in residue:
                    c_alpha = residue['CA'].coord
                    sc_x, sc_y, sc_z = calculate_center_of_mass(residue)
                    ss = stored.ss_values[i]

                    # Check interactions using atom-atom distances
                    interaction_data = {aa: 0 for aa in ['G', 'A', 'V', 'I', 'L', 'S', 'T', 'D', 'N', 'E', 'Q', 'K', 'R', 'C', 'M', 'F', 'Y', 'W', 'H', 'P']}
                    for ab_residue in antibody_residues:
                        if check_atom_atom_interaction(residue, ab_residue):
                            one_letter_code = PDB.Polypeptide.protein_letters_3to1[ab_residue.resname]
                            interaction_data[one_letter_code] = 1

                    data = {'name': '1ors_B_A_C',
                            'ResID': residue.id[1],  # Residue number
                            'AA': residue.resname,
                            'SS': ss,
                            'Cax': c_alpha[0],
                            'Cay': c_alpha[1],
                            'Caz': c_alpha[2],
                            'SCx': sc_x,
                            'SCy': sc_y,
                            'SCz': sc_z,
                            'SaSaRate': sasa_values[i]}
                    data.update(interaction_data)
                    df = pd.concat([df, pd.DataFrame([data])], ignore_index=True)

# Save DataFrame to TSV
output_path = "/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_info/1ors_B_A_C_info.tsv"
df.to_csv(output_path, sep='\t', index=False)


In [ ]:
df.tail(10)

In [ ]:
#@title TEST - generate the antigen info data frame with COM of sidechain-sidechain interaction
# this antigen data frame contains the structure info and interaction info

from Bio import PDB
import pandas as pd
from pymol import cmd, stored
from scipy.spatial import distance
import os

# Function to calculate center of mass of sidechain
def calculate_center_of_mass(residue):
    mass = 0
    x, y, z = 0, 0, 0
    backbone_atoms = ['N', 'CA', 'C', 'O']
    for atom in residue.get_unpacked_list():
        if atom.name not in backbone_atoms:
            mass += atom.mass
            x += atom.coord[0] * atom.mass
            y += atom.coord[1] * atom.mass
            z += atom.coord[2] * atom.mass
    if mass == 0:
        return 0, 0, 0
    return x / mass, y / mass, z / mass

# Load the PDB file into PyMOL
pdb_file_path = "/content/drive/MyDrive/@antibody/antigen_epitope/data/structures/1ors_B_A_C.pdb"
cmd.load(pdb_file_path, "protein_complex")

# Calculate relative SASA
SaSaRate = cmd.get_sasa_relative("protein_complex and chain C")
sasa_values = [value for key, value in SaSaRate.items()]

# Initialize an empty list for secondary structure
stored.ss_values = []
cmd.iterate("protein_complex and chain C and name CA", "stored.ss_values.append(ss)")

# Initialize PDB Parser
parser = PDB.PDBParser(QUIET=True)

# Load structure
structure = parser.get_structure('1ors_B_A_C', pdb_file_path)

# Initialize DataFrame
columns = ['name', 'ResID', 'AA', 'SS', 'Cax', 'Cay', 'Caz', 'SCx', 'SCy', 'SCz', 'SaSaRate',
           'G', 'A', 'V', 'I', 'L', 'S', 'T', 'D', 'N', 'E', 'Q', 'K', 'R', 'C', 'M', 'F', 'Y', 'W', 'H', 'P']
df = pd.DataFrame(columns=columns)

# Prepare a list for antibody sidechain centers
antibody_centers = []

# Populate antibody_centers
for model in structure:
    for chain in model:
        if chain.id in ["B", "A"]:
            for residue in chain:
                if "CA" in residue:
                    sc_x, sc_y, sc_z = calculate_center_of_mass(residue)
                    ab_resname = residue.resname
                    antibody_centers.append((ab_resname, (sc_x, sc_y, sc_z)))

# Iterate through structure to get antigen residues
for model in structure:
    for chain in model:
        if chain.id == "C":
            for i, residue in enumerate(chain):
                if "CA" in residue:
                    c_alpha = residue['CA'].coord
                    sc_x, sc_y, sc_z = calculate_center_of_mass(residue)
                    ss = stored.ss_values[i]

                    interaction_data = {aa: 0 for aa in ['G', 'A', 'V', 'I', 'L', 'S', 'T', 'D', 'N', 'E', 'Q', 'K', 'R', 'C', 'M', 'F', 'Y', 'W', 'H', 'P']}
                    antigen_center = (sc_x, sc_y, sc_z)

                    for ab_resname, ab_center in antibody_centers:
                        if distance.euclidean(antigen_center, ab_center) <= 7:
                            #one_letter_code = PDB.Polypeptide.three_to_one(ab_resname)
                            one_letter_code = PDB.Polypeptide.protein_letters_3to1[ab_resname]
                            interaction_data[one_letter_code] = 1

                    data = {'name': '1ors_B_A_C',
                            'ResID': residue.id[1],  # Residue number
                            'AA': residue.resname,
                            'SS': ss,
                            'Cax': c_alpha[0],
                            'Cay': c_alpha[1],
                            'Caz': c_alpha[2],
                            'SCx': sc_x,
                            'SCy': sc_y,
                            'SCz': sc_z,
                            'SaSaRate': sasa_values[i]}
                    data.update(interaction_data)
                    df = pd.concat([df, pd.DataFrame([data])], ignore_index=True)

# Save DataFrame to TSV
output_path = "/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_info/1ors_B_A_C_info.tsv"
df.to_csv(output_path, sep='\t', index=False)

In [ ]:
!apt-get install -y python3-pymol

In [ ]:
#@title TEST -- extract antigen chain and calculate its SaSaRate
# Import the PyMOL command module
from pymol import cmd

# Load the PDB file into PyMOL
pdb_file_path = "/content/drive/MyDrive/@antibody/antigen_epitope/data/structures/7xrz_H_L_X.pdb"
cmd.load(pdb_file_path, "protein_complex")

# Select chain C
cmd.select("chainX", "protein_complex and chain X")

# Calculate the relative SASA
SaSaRate = cmd.get_sasa_relative("chainX")
print(SaSaRate)

sasa_values = []
# Iterate over the defaultdict to get just the float values
for key, value in SaSaRate.items():
    sasa_values.append(value)
print(sasa_values)

# Iterate over the defaultdict to get just the float values
for key, sasa_value in SaSaRate.items():
    print(key, sasa_value)

size_of_dictionary = len(SaSaRate)
print(f"The size of the SaSaRate dictionary is {size_of_dictionary}")

In [ ]:
#@title TEST -- secondary structure
from pymol import cmd, stored

# Initialize an empty list to store secondary structure data in the stored namespace
stored.ss_values = []

# Load the PDB file into PyMOL
pdb_file_path = "/content/drive/MyDrive/@antibody/antigen_epitope/data/structures/7xrz_H_L_X.pdb"
cmd.load(pdb_file_path, "protein_complex")

# Select chain X
cmd.select("chainX", "protein_complex and chain X")

# Use cmd.iterate to access the 'ss' (secondary structure) attribute of each residue
cmd.iterate("chainX and name CA", "stored.ss_values.append((resi, ss))")

# Print the secondary structure for each residue
#for resi, ss in stored.ss_values:
#    print(f"Residue {resi}: Secondary Structure = {ss}")

stored.ss_values = []
cmd.iterate("protein_complex and chain X and name CA", "stored.ss_values.append(ss)")
print(stored.ss_values)

In [ ]:
#@title TEST -- extract the antigen chain C and add SaSaRate to the b-factor of each residue

from Bio.PDB import PDBParser, PDBIO

# Initialize the PDB parser and read the file
parser = PDBParser(QUIET=True)
structure = parser.get_structure('protein_complex', '/content/drive/MyDrive/@antibody/antigen_epitope/data/structures/8goc_M_N_C.pdb')

# Create an empty Structure to save the chain C
from Bio.PDB.Structure import Structure
chain_C_structure = Structure('chain_C_structure')

# Add a model to the new structure
from Bio.PDB.Model import Model
model = Model(0)
chain_C_structure.add(model)

# Get chain C from the original structure and add it to the new structure
chain_C = structure[0]['C']
model.add(chain_C)

# Update B-factors based on the SaSaRate (you should already have this from your PyMOL calculations)
# Here, we're assuming SaSaRate is a dictionary with keys being the residue numbers and values being the calculated SASA
# SaSaRate = {...}

for residue in chain_C:
    resi = residue.get_id()[1]
    sasa_value = SaSaRate.get(('protein_complex', '', 'C', str(resi)), None)
    if sasa_value is not None:
        for atom in residue:
            atom.set_bfactor(sasa_value)

# Save the new structure as a PDB file
io = PDBIO()
io.set_structure(chain_C_structure)
io.save('/content/drive/MyDrive/@antibody/antigen_epitope/data/structures_antigen/8goc_C.pdb')
